<font color="orangered">
    
## RAG Pipelines- Data Ingestion to Vector DB Pipeline
</font>

<font color="orangered">
    
### Step 1: load the data
- Read all the pdf's inside the directory
</font>

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter #from langchain.text_splitter import RecursiveCharacterTextSplitter  (this one does not work)
from pathlib import Path

In [49]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}") 
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: MemoryBasedTemporalNetworkPrediction.pdf
  ✓ Loaded 13 pages

Processing: short and long term prediction.pdf
  ✓ Loaded 25 pages

Processing: Temporal_Network_Prediction_and_Interpretation.pdf
  ✓ Loaded 10 pages

Processing: Vulnerability_of_Information_Transport_on_Temporal_Networks_to_Link_Removal.pdf
  ✓ Loaded 10 pages

Total documents loaded: 58


In [50]:
(type(all_pdf_documents),len(all_pdf_documents))

(list, 58)

In [51]:
all_pdf_documents

[Document(metadata={'producer': 'Acrobat Distiller 10.1.8 (Windows)', 'creator': 'Springer', 'creationdate': '2023-01-21T17:03:28+05:30', 'author': 'Li Zou', 'crossmarkdomains[1]': 'springer.com', 'crossmarkdomains[2]': 'springerlink.com', 'crossmarkdomainexclusive': 'true', 'crossmarkmajorversiondate': '2010-04-23', 'keywords': 'Temporal network prediction; Network-Based prediction; Temporal network property', 'moddate': '2023-01-21T03:39:53-08:00', 'subject': 'Complex Networks and Their Applications XI, doi:10.1007/978-3-031-21131-7_51', 'title': 'Memory Based Temporal Network Prediction', 'doi': '10.1007/978-3-031-21131-7_51', 'robots': 'noindex', 'source': '..\\data\\pdf\\MemoryBasedTemporalNetworkPrediction.pdf', 'total_pages': 13, 'page': 0, 'page_label': '661', 'source_file': 'MemoryBasedTemporalNetworkPrediction.pdf', 'file_type': 'pdf'}, page_content='Memory Based Temporal Network\nPrediction\nLi Zou, An Wang, and Huijuan Wang\nAbstract Temporal networks are networks like phys

<font color="orangered">
    
### Step 2: chunking the data
- Text splitting get into chunksText splitting get into chunks
</font>

In [52]:
### Text splitting get into chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""])
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs
chunks=split_documents(all_pdf_documents)

Split 58 documents into 276 chunks

Example chunk:
Content: Memory Based Temporal Network
Prediction
Li Zou, An Wang, and Huijuan Wang
Abstract Temporal networks are networks like physical contact networks whose
topology changes over time. Predicting future te...
Metadata: {'producer': 'Acrobat Distiller 10.1.8 (Windows)', 'creator': 'Springer', 'creationdate': '2023-01-21T17:03:28+05:30', 'author': 'Li Zou', 'crossmarkdomains[1]': 'springer.com', 'crossmarkdomains[2]': 'springerlink.com', 'crossmarkdomainexclusive': 'true', 'crossmarkmajorversiondate': '2010-04-23', 'keywords': 'Temporal network prediction; Network-Based prediction; Temporal network property', 'moddate': '2023-01-21T03:39:53-08:00', 'subject': 'Complex Networks and Their Applications XI, doi:10.1007/978-3-031-21131-7_51', 'title': 'Memory Based Temporal Network Prediction', 'doi': '10.1007/978-3-031-21131-7_51', 'robots': 'noindex', 'source': '..\\data\\pdf\\MemoryBasedTemporalNetworkPrediction.pdf', 'total_pages': 13

In [53]:
(type(chunks),len(chunks))

(list, 276)

In [54]:
chunks[0]

Document(metadata={'producer': 'Acrobat Distiller 10.1.8 (Windows)', 'creator': 'Springer', 'creationdate': '2023-01-21T17:03:28+05:30', 'author': 'Li Zou', 'crossmarkdomains[1]': 'springer.com', 'crossmarkdomains[2]': 'springerlink.com', 'crossmarkdomainexclusive': 'true', 'crossmarkmajorversiondate': '2010-04-23', 'keywords': 'Temporal network prediction; Network-Based prediction; Temporal network property', 'moddate': '2023-01-21T03:39:53-08:00', 'subject': 'Complex Networks and Their Applications XI, doi:10.1007/978-3-031-21131-7_51', 'title': 'Memory Based Temporal Network Prediction', 'doi': '10.1007/978-3-031-21131-7_51', 'robots': 'noindex', 'source': '..\\data\\pdf\\MemoryBasedTemporalNetworkPrediction.pdf', 'total_pages': 13, 'page': 0, 'page_label': '661', 'source_file': 'MemoryBasedTemporalNetworkPrediction.pdf', 'file_type': 'pdf'}, page_content='Memory Based Temporal Network\nPrediction\nLi Zou, An Wang, and Huijuan Wang\nAbstract Temporal networks are networks like physi

In [55]:
chunks[0].metadata

{'producer': 'Acrobat Distiller 10.1.8 (Windows)',
 'creator': 'Springer',
 'creationdate': '2023-01-21T17:03:28+05:30',
 'author': 'Li Zou',
 'crossmarkdomains[1]': 'springer.com',
 'crossmarkdomains[2]': 'springerlink.com',
 'crossmarkdomainexclusive': 'true',
 'crossmarkmajorversiondate': '2010-04-23',
 'keywords': 'Temporal network prediction; Network-Based prediction; Temporal network property',
 'moddate': '2023-01-21T03:39:53-08:00',
 'subject': 'Complex Networks and Their Applications XI, doi:10.1007/978-3-031-21131-7_51',
 'title': 'Memory Based Temporal Network Prediction',
 'doi': '10.1007/978-3-031-21131-7_51',
 'robots': 'noindex',
 'source': '..\\data\\pdf\\MemoryBasedTemporalNetworkPrediction.pdf',
 'total_pages': 13,
 'page': 0,
 'page_label': '661',
 'source_file': 'MemoryBasedTemporalNetworkPrediction.pdf',
 'file_type': 'pdf'}

<font color="orangered">
    
### Step 3: embedding and vectorDB

</font>

In [56]:
import numpy as np
from sentence_transformers import SentenceTransformer # embedding model
import chromadb  # for vector store
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [57]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """ Initialize the embedding manager
            Args:
                model_name: HuggingFace model name for sentence embeddings"""
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
            Args:
                texts: List of text strings to embed
            Returns:
                numpy array of embeddings with shape (len(texts), embedding_dim) """
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=False)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
## initialize the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension: 384


<font color="orangered">
    
### Step 4: vector store

</font>

In [60]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """Initialize the vector store
           Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store"""
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"})
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}") 
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embeddings to the vector store
           Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding {len(documents)} documents to vector store...")
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            # Document content
            documents_text.append(doc.page_content)
            # Embedding
            embeddings_list.append(embedding.tolist())
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text)
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}") 
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 241


In [61]:
chunks

[Document(metadata={'producer': 'Acrobat Distiller 10.1.8 (Windows)', 'creator': 'Springer', 'creationdate': '2023-01-21T17:03:28+05:30', 'author': 'Li Zou', 'crossmarkdomains[1]': 'springer.com', 'crossmarkdomains[2]': 'springerlink.com', 'crossmarkdomainexclusive': 'true', 'crossmarkmajorversiondate': '2010-04-23', 'keywords': 'Temporal network prediction; Network-Based prediction; Temporal network property', 'moddate': '2023-01-21T03:39:53-08:00', 'subject': 'Complex Networks and Their Applications XI, doi:10.1007/978-3-031-21131-7_51', 'title': 'Memory Based Temporal Network Prediction', 'doi': '10.1007/978-3-031-21131-7_51', 'robots': 'noindex', 'source': '..\\data\\pdf\\MemoryBasedTemporalNetworkPrediction.pdf', 'total_pages': 13, 'page': 0, 'page_label': '661', 'source_file': 'MemoryBasedTemporalNetworkPrediction.pdf', 'file_type': 'pdf'}, page_content='Memory Based Temporal Network\nPrediction\nLi Zou, An Wang, and Huijuan Wang\nAbstract Temporal networks are networks like phys

In [62]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)

##store in the vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 276 texts...
Generated embeddings with shape: (276, 384)
Adding 276 documents to vector store...
Successfully added 276 documents to vector store
Total documents in collection: 517


In [63]:
texts

['Memory Based Temporal Network\nPrediction\nLi Zou, An Wang, and Huijuan Wang\nAbstract Temporal networks are networks like physical contact networks whose\ntopology changes over time. Predicting future temporal network is crucial e.g., to\nforecast and mitigate the spread of epidemics and misinformation on the network.\nMost existing methods for temporal network prediction are based on machine learn-\ning algorithms, at the expense of high computational costs and limited interpretation\nof the underlying mechanisms that form the networks. This motivates us to develop\nnetwork-based models to predict the temporal network at the next time step based on\nthe network observed in the past. Firstly, we investigate temporal network properties\nto motivate our network prediction models and to explain how the performance of\nthese models depends on the temporal networks. We explore the similarity between\nthe network topology (snapshot) at any two time steps with a given time lag/interval.',


<font color="orangered">
    
### Step 5: Retriever Pipeline From VectorStore

</font>

In [65]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """Initialize the retriever
           Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings"""
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve relevant documents for a query
           Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold  
           Returns: List of dictionaries containing retrieved documents and metadata """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(query_embeddings=[query_embedding.tolist()], n_results=top_k)
            
            # Process results
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1})
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            return retrieved_docs  
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
            
rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [68]:
rag_retriever.retrieve("What is temporal networks")

Retrieving documents for query: 'What is temporal networks'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_6b3ff9e0_3',
  'content': '662 L. Zou et al.\n1 Introduction\nComplex systems can be represented as networks, where nodes represent the compo-\nnents of a system and links denote the interaction or relation between the components.\nThe interactions are, in many cases, not continuously active. For example, individuals\nconnect via email, phone call or physical contact at speciﬁc times instead of con-\nstantly. Temporal networks [ 1, 2] could represent these systems more realistically\nwith time-varying network topology. It has been shown that temporal network prop-\nerties such as community structure, the degree distribution in the aggregated network,\ninter-event time and the non-Markovian evolution inﬂuence dynamic processes on\nthe temporal network [ 3–6].\nTemporal network prediction is a task of predicting temporal contacts at the next\ntime step based on the temporal network topology observed in the previous L steps.\nPredicting the temporal network such as a physical

<font color="orangered">
    
### Step 6: RAG Pipeline- VectorDB To LLM Output Generatio

</font>

In [71]:
from langchain_openai import ChatOpenAI  #from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate #from langchain.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage #from langchain.schema import HumanMessage, SystemMessage
import os
from dotenv import load_dotenv
load_dotenv()

In [75]:
class OpenAILLM:
    def __init__(self, model_name: str = "gpt-3.5-turbo", api_key: str =None):
        """Initialize OpenAI LLM
           Args:
            model_name: OpenAI model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: openai API key (or set OPENAI_API_KEY environment variable) """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("OPENAI_API_KEY")
        
        if not self.api_key: raise ValueError("openai API key is required. Set OPENAI_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatOpenAI(openai_api_key=self.api_key, model=self.model_name, temperature=0.1)
        print(f"Initialized OpenAI LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """Generate response using retrieved context
           Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length  
           Returns: Generated response string """
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content  
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [79]:
# Initialize openai LLM (you'll need to set OPENAI_API_KEY environment variable)
try:
    openai_llm = OpenAILLM(api_key=os.getenv("OPENAI_API_KEY"))
    print("OpenAI LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your OPENAI_API_KEY environment variable to use the LLM.")
    openai_llm = None

### get the context from the retriever and pass it to the LLM
rag_retriever.retrieve("waht is temporal networks")

Initialized OpenAI LLM with model: gpt-3.5-turbo
OpenAI LLM initialized successfully!
Retrieving documents for query: 'waht is temporal networks'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_6b3ff9e0_3',
  'content': '662 L. Zou et al.\n1 Introduction\nComplex systems can be represented as networks, where nodes represent the compo-\nnents of a system and links denote the interaction or relation between the components.\nThe interactions are, in many cases, not continuously active. For example, individuals\nconnect via email, phone call or physical contact at speciﬁc times instead of con-\nstantly. Temporal networks [ 1, 2] could represent these systems more realistically\nwith time-varying network topology. It has been shown that temporal network prop-\nerties such as community structure, the degree distribution in the aggregated network,\ninter-event time and the non-Markovian evolution inﬂuence dynamic processes on\nthe temporal network [ 3–6].\nTemporal network prediction is a task of predicting temporal contacts at the next\ntime step based on the temporal network topology observed in the previous L steps.\nPredicting the temporal network such as a physical

<font color="orangered">
    
### Step 7: Integration Vectordb Context pipeline With LLM output

</font>

In [73]:
### Simple RAG pipeline with OpenAI LLM
from langchain_openai import ChatOpenAI   #from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the OpenAI LLM (set your OPENAI_API_KEY in environment)
groq_api_key = os.getenv("OPENAI_API_KEY")
llm=ChatOpenAI(openai_api_key=groq_api_key,model="gpt-3.5-turbo",temperature=0.1)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context: return "No relevant context found to answer the question."
    
    ## generate the answwer using OPENAI LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content
answer=rag_simple("What is a temporal network?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is a temporal network?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
A temporal network is a network where nodes represent components and links represent interactions between components, with interactions occurring at specific time stamps rather than constantly.


In [33]:
### get the context from the retriever and pass it to the LLM
rag_retriever.retrieve("waht is temporal networks")

Retrieving documents for query: 'waht is temporal networks'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 58.83it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_af647e49_106',
  'content': 'works, where nodes denote the components and links\ndenote relations or interaction between these components. In\nmany cases, however, the interactions are not continuously\nactive. For example, individuals connect via email, text mes-\nsage, phone call or physical contact at speciﬁc time stamps\ninstead of constantly. Temporal networks [1]–[3] could repre-\nsent these systems more realistically with time-varying net-\nwork topology. A temporal network can be regarded as a\nstatic network where each link is further associated with a\ntime series specifying whether an interaction (contact) occurs\nor not at each time step. Temporal networks display non-trivial\nproperties, which may have profound effect on the dynamic\nprocesses deployed on them. For example, the inter-event (con-\ntact, activation) time between a node pair has been found to fol-\nlow a heavy-tail or power-law distribution in many temporal\nnetworks [4]–[6]. It has been shown th

<font color="orangered">
    
### Step 7: Enhanced RAG Pipeline Features

</font>

In [83]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """ RAG pipeline with extra features:
       - Returns answer, sources, confidence score, and optionally full context."""
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results: return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {'answer': response.content, 'sources': sources, 'confidence': confidence}
    if return_context: output['context'] = context
    return output

# Example usage:
result = rag_advanced("what is temporal networks", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'what is temporal networks'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: Temporal networks are networks where nodes represent components of a system and links denote interactions or relations between components that are not continuously active, but occur at specific times.
Sources: [{'source': 'MemoryBasedTemporalNetworkPrediction.pdf', 'page': 1, 'score': 0.4819400906562805, 'preview': '662 L. Zou et al.\n1 Introduction\nComplex systems can be represented as networks, where nodes represent the compo-\nnents of a system and links denote the interaction or relation between the components.\nThe interactions are, in many cases, not continuously active. For example, individuals\nconnect via ...'}, {'source': 'Temporal_Network_Prediction_and_Interpretation.pdf', 'page': 0, 'score': 0.4493083953857422, 'preview': 'works, where nodes denote the compo

In [84]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what are temporal networks", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what are temporal networks'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
662 L. Zou et al.
1 Introduction
Complex systems can be represented as networks, where nodes represent the compo-
nents of a system and links denote the interaction or relation between the components.
The interactions are, in many cases, not continuously active. For example, individuals
connect via email, phone call or physical contact at speciﬁc times instead of con-
stantly. Temporal networks [ 1, 2] could represent these systems more realistically
with time-varying network topology. It has been shown that temporal network prop-
erties such as community structure, the degree distribution in the aggregated network,
inter-event time and the non-Markovian evolution inﬂuence dynamic processes on